# 🗣️ VoxCPM2 (2B) — Zero-Shot Voice Cloning

Clone a voice from a short reference clip and make it speak a hard-coded
sentence, using **`openbmb/VoxCPM2`** — a **2B-parameter, tokenizer-free**
Text-to-Speech system that outputs **48 kHz** studio-quality audio.

**How it works (the honest version).** Unlike Orpheus (a token-completion
Speech-LLM that needs SNAC codec tokens and a hand-built two-turn prompt),
VoxCPM2 exposes a **high-level cloning API**. You hand it the target text plus a
reference clip; the model encodes the reference internally (its own audio VAE),
conditions on it, and synthesizes the target text in that voice. There is no
manual token layout to manage.

VoxCPM2 supports two cloning modes — both are in the config:

1. **Simple cloning** — pass only `reference_wav_path`. No transcript needed; the
   model infers timbre/style straight from the audio.
2. **“Ultimate” cloning** — also pass `prompt_wav_path` + `prompt_text` (the exact
   words spoken in the clip). The aligned transcript acts as an in-context
   prompt and usually gives a tighter, more faithful clone. This is the mode
   that parallels Orpheus's `REF_TRANSCRIPT` approach, so it's the default here.

> ⚠️ **Limitations — read these.** Clone quality depends on a **clean** reference
> (single speaker, no music/noise) and — for ultimate mode — a `PROMPT_TEXT`
> that matches the words actually spoken. VoxCPM2 clones well from fairly short
> clips (~5–30 s); longer is not always better. Expect to try a couple of
> reference clips. You can also enable the built-in denoiser (`LOAD_DENOISER`)
> to clean a noisy reference.

---
## ✋ Consent & responsible use
Only clone voices you **own** or have **explicit permission** to clone. Do not
use this to impersonate, deceive, or create misleading content. Cloning a
person's voice without consent may be illegal in your jurisdiction.

---
## ▶️ Running this in Google Colab
1. **Runtime → Change runtime type → GPU** (a T4 is plenty — VoxCPM2 needs ~8 GB
   VRAM). It also runs on CPU / Apple `mps`, just slowly.
2. Get the notebook into Colab — any one of:
   - **File → Upload notebook** and pick this `.ipynb`, **or**
   - **File → Open notebook → GitHub** tab, paste your repo URL, **or**
   - clone the repo from a cell: `!git clone https://github.com/<you>/<repo>.git`
     then open the `.ipynb` from the file browser on the left.
3. Run the cells **top to bottom**. The first install + model download take a
   few minutes.

## 1. Setup and installation
Installs the `voxcpm` package (which pulls a compatible torch/transformers) plus
audio I/O libraries. Safe to re-run. On Colab `ffmpeg` is preinstalled (needed
for MP3/M4A decoding).

> Requirements per the project: **Python ≥ 3.10 (< 3.13)**, **PyTorch ≥ 2.5.0**,
> **CUDA ≥ 12.0** for GPU. Colab satisfies these.

In [ ]:
# --- Install dependencies (Colab or fresh local env) ---------------------
# %%capture  # uncomment to hide pip noise
import sys, subprocess, shutil

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# The voxcpm package brings its own compatible torch/transformers stack.
pip_install("voxcpm")
# Audio I/O (soundfile writes the output WAV; librosa decodes/preps the ref).
pip_install("soundfile", "librosa")

# ffmpeg is required to decode MP3/M4A. It ships with Colab; warn if missing.
if shutil.which("ffmpeg") is None:
    print("⚠️ ffmpeg not found. WAV will still work, but MP3/M4A may not.")
    print("   Colab: run  !apt-get -y install ffmpeg")
    print("   macOS: brew install ffmpeg   |   Ubuntu: sudo apt install ffmpeg")
else:
    print("✅ ffmpeg found:", shutil.which("ffmpeg"))
print("✅ Install step finished.")

## 2. Imports & device
VoxCPM2 runs on CUDA, Apple `mps`, or CPU. Unlike the Orpheus notebook we do
**not** hard-require a GPU — but a GPU is strongly recommended (CPU synthesis is
very slow).

In [ ]:
import os, gc, warnings
import numpy as np
import torch

warnings.filterwarnings("ignore")

# --- Pick the best available device -------------------------------------------
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("✅ GPU:", torch.cuda.get_device_name(0))
    print("   VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("✅ Apple MPS device (no CUDA). Synthesis will be slower than a GPU.")
else:
    DEVICE = "cpu"
    print("⚠️ No GPU/MPS detected — running on CPU. This will be SLOW.")
    print("   In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU.")

print("   torch:", torch.__version__, "| device:", DEVICE)

## 3. Configuration
Everything you might change lives here.

For **ultimate** cloning (the default) you should set `PROMPT_TEXT` to the exact
words spoken in your reference clip — accurate text improves the clone. For
**simple** cloning, `PROMPT_TEXT` is ignored.

In [ ]:
# --- Model -------------------------------------------------------------------
MODEL_ID      = "openbmb/VoxCPM2"   # 2B tokenizer-free TTS
LOAD_DENOISER = False               # Load the ZipEnhancer denoiser (~extra DL/VRAM).
                                    # Set True if your reference clip is noisy AND
                                    # set DENOISE=True below to apply it.

# --- Cloning mode ------------------------------------------------------------
# "ultimate" -> uses prompt_wav_path + PROMPT_TEXT + reference_wav_path (best).
# "simple"   -> uses reference_wav_path only (no transcript needed).
CLONE_MODE    = "ultimate"

# --- Reference voice ---------------------------------------------------------
REF_AUDIO_PATH = ""     # leave "" to upload in Colab, or set a local path here
# ⬇️ EXACTLY what is said in the reference clip (used only in "ultimate" mode):
PROMPT_TEXT    = "Replace this text with the exact words spoken in your reference audio sample."

# --- What the cloned voice should say (hard-coded, per requirements) ----------
TARGET_TEXT = ("Hello, this is a cloned voice test using VoxCPM2. The model "
               "should speak this sentence in the style of the uploaded voice sample.")

# --- Generation knobs (VoxCPM2 generate() params) ----------------------------
CFG_VALUE           = 2.0    # classifier-free guidance strength (default 2.0).
                             # Higher = follows the prompt/style more strongly.
INFERENCE_TIMESTEPS = 10     # diffusion steps (default 10). More = slightly
                             # higher quality, slower.
DENOISE             = False  # denoise the reference before cloning. Needs
                             # LOAD_DENOISER=True above.
NORMALIZE           = False  # text normalization (numbers/abbreviations). Leave
                             # False unless your target text has lots of digits.

# --- Audio preprocessing (light — VoxCPM2 also resamples internally) ----------
REF_TARGET_SR  = 16000   # VoxCPM2 accepts a 16 kHz mono reference
TRIM_TOP_DB    = 30      # silence-trim aggressiveness (higher = less trimming)
PEAK_NORM      = 0.95    # safe peak normalization target
IDEAL_MIN_SEC  = 5       # VoxCPM2 clones fine from short clips
IDEAL_MAX_SEC  = 30

OUTPUT_PATH = "output_cloned_voice.wav"
assert CLONE_MODE in ("ultimate", "simple"), "CLONE_MODE must be 'ultimate' or 'simple'"
print(f"Config loaded. Mode={CLONE_MODE}. Target text:\n  {TARGET_TEXT}")

## 4. Upload / load voice sample
In Colab a file picker appears. Locally (or to skip the picker), set
`REF_AUDIO_PATH` in the config cell. Accepts WAV/MP3/M4A/FLAC.

In [ ]:
# Detect Colab and offer an upload widget if no path was configured.
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if not REF_AUDIO_PATH:
    if IN_COLAB:
        print("Upload your voice sample (WAV/MP3/M4A/FLAC, ~5-30s):")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        REF_AUDIO_PATH = list(uploaded.keys())[0]
    else:
        raise RuntimeError(
            "REF_AUDIO_PATH is empty and not running in Colab.\n"
            "Set REF_AUDIO_PATH in the Configuration cell to your audio file."
        )

if not os.path.exists(REF_AUDIO_PATH):
    raise FileNotFoundError(f"Audio file not found: {REF_AUDIO_PATH}")
print("✅ Reference audio:", REF_AUDIO_PATH)

## 5. Audio preprocessing (light)
VoxCPM2 loads and resamples the reference internally, so heavy preprocessing
isn't required. We do a light cleanup anyway — convert to **mono @ 16 kHz**, trim
leading/trailing silence, peak-normalize — and write a clean WAV that we pass to
the model. This also lets us validate the duration and preview the clip.

In [ ]:
import librosa
import soundfile as sf
from IPython.display import Audio, display

def load_and_preprocess(path):
    """Return a cleaned float32 mono waveform at REF_TARGET_SR."""
    try:
        wav, sr = librosa.load(path, sr=REF_TARGET_SR, mono=True)
    except Exception as e:
        raise RuntimeError(
            f"Could not decode '{path}'. Unsupported format or missing ffmpeg.\n"
            f"Original error: {e}"
        )
    # Trim silence from both ends.
    wav, _ = librosa.effects.trim(wav, top_db=TRIM_TOP_DB)
    # Safe peak normalization (avoid divide-by-zero on silent clips).
    peak = float(np.max(np.abs(wav))) if wav.size else 0.0
    if peak > 0:
        wav = (wav / peak) * PEAK_NORM
    return wav.astype(np.float32), REF_TARGET_SR

ref_wav, ref_sr = load_and_preprocess(REF_AUDIO_PATH)
duration = len(ref_wav) / ref_sr

# Save the cleaned reference; this is the path we hand to VoxCPM2.
CLEAN_REF_PATH = "reference_clean.wav"
sf.write(CLEAN_REF_PATH, ref_wav, ref_sr)
print(f"Processed: {duration:.1f}s, mono, {ref_sr} Hz -> {CLEAN_REF_PATH}")

if duration < 2:
    print(f"🚨 WARNING: only {duration:.1f}s — very short. Cloning will be poor.")
elif duration < IDEAL_MIN_SEC:
    print(f"⚠️ {duration:.1f}s is a little short; {IDEAL_MIN_SEC}-{IDEAL_MAX_SEC}s is ideal.")
elif duration > IDEAL_MAX_SEC:
    print(f"⚠️ {duration:.1f}s is longer than {IDEAL_MAX_SEC}s; fine, but not necessary.")
else:
    print("✅ Duration is in a good range for cloning.")

print("Preview of your processed reference:")
display(Audio(ref_wav, rate=ref_sr))

## 6. Model loading
Downloads and loads VoxCPM2 via the high-level `VoxCPM.from_pretrained` API. The
first run downloads the 2B weights (a few minutes). Includes an out-of-memory
fallback hint.

In [ ]:
from voxcpm import VoxCPM

try:
    model = VoxCPM.from_pretrained(
        MODEL_ID,
        load_denoiser=LOAD_DENOISER,   # ZipEnhancer denoiser (optional)
        device=DEVICE,                 # "cuda" / "mps" / "cpu"
    )
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache(); gc.collect()
    raise RuntimeError(
        "CUDA OOM while loading VoxCPM2.\n"
        "Fixes: set LOAD_DENOISER=False, restart the runtime, or use a bigger GPU."
    )
except Exception as e:
    raise RuntimeError(
        f"Model download/load failed: {e}\n"
        "If the model is gated, run `huggingface-cli login` (or set HF_TOKEN) and "
        "accept the model terms on its Hugging Face page."
    )

SAMPLE_RATE = model.tts_model.sample_rate  # VoxCPM2 outputs 48 kHz
print(f"✅ VoxCPM2 ready. Output sample rate: {SAMPLE_RATE} Hz")

## 7. Voice cloning / generation
Calls `model.generate(...)`. In **ultimate** mode we pass the reference clip as
both the in-context prompt (`prompt_wav_path` + `prompt_text`) and the timbre
`reference_wav_path`. In **simple** mode we pass only `reference_wav_path`.

The call returns a float32 waveform (a NumPy array) at `SAMPLE_RATE`.

In [ ]:
gen_kwargs = dict(
    text=TARGET_TEXT,
    reference_wav_path=CLEAN_REF_PATH,
    cfg_value=CFG_VALUE,
    inference_timesteps=INFERENCE_TIMESTEPS,
    normalize=NORMALIZE,
    denoise=DENOISE,
)

if CLONE_MODE == "ultimate":
    # Add the aligned transcript prompt for a tighter clone.
    gen_kwargs["prompt_wav_path"] = CLEAN_REF_PATH
    gen_kwargs["prompt_text"] = PROMPT_TEXT
    if "Replace this text" in PROMPT_TEXT:
        print("⚠️ PROMPT_TEXT is still the placeholder. For best results, set it to "
              "the exact words spoken in your clip (or switch CLONE_MODE='simple').")

try:
    audio_np = model.generate(**gen_kwargs)
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache(); gc.collect()
    raise RuntimeError(
        "CUDA OOM during generation.\n"
        "Fixes: use a shorter reference clip, lower INFERENCE_TIMESTEPS, "
        "set LOAD_DENOISER=False, or restart the runtime."
    )

audio_np = np.asarray(audio_np, dtype=np.float32).squeeze()
print(f"✅ Generated {len(audio_np)/SAMPLE_RATE:.1f}s of audio "
      f"({len(audio_np)} samples @ {SAMPLE_RATE} Hz).")

## 8. Save and play output audio
Saves `output_cloned_voice.wav` at the model's native 48 kHz and shows a player.

In [ ]:
import soundfile as sf
sf.write(OUTPUT_PATH, audio_np, SAMPLE_RATE)
print(f"✅ Saved {OUTPUT_PATH}  ({len(audio_np)/SAMPLE_RATE:.1f}s @ {SAMPLE_RATE} Hz)")

from IPython.display import Audio, display
print("🔊 Cloned voice saying the target text:")
display(Audio(audio_np, rate=SAMPLE_RATE))

# In Colab, uncomment to download:
# from google.colab import files; files.download(OUTPUT_PATH)

## 9. Optional extras — style control & voice design (no reference)

VoxCPM2 can also **steer** the cloned voice with a natural-language style tag at
the start of the text (in parentheses), and can **design a brand-new voice** from
a description alone — no reference clip required.

> These are bonus capabilities VoxCPM2 has that Orpheus does not. They are
> independent of the clone above; set the flag and run.

In [ ]:
# ====== OPTIONAL — style-controlled clone + voice design (run only if needed) ===
RUN_EXTRAS = False   # set True to execute this cell's body

if RUN_EXTRAS:
    # (a) Style-controlled cloning: a (parenthetical) tag steers emotion/pace
    #     while keeping the cloned timbre.
    styled = model.generate(
        text="(slightly faster, cheerful tone)" + TARGET_TEXT,
        reference_wav_path=CLEAN_REF_PATH,
        cfg_value=CFG_VALUE,
        inference_timesteps=INFERENCE_TIMESTEPS,
    )
    styled = np.asarray(styled, dtype=np.float32).squeeze()
    sf.write("output_styled_clone.wav", styled, SAMPLE_RATE)
    print("✅ Saved output_styled_clone.wav (cloned voice, cheerful/faster)")
    display(Audio(styled, rate=SAMPLE_RATE))

    # (b) Voice design: brand-new voice from a description — NO reference audio.
    designed = model.generate(
        text="(A young woman, gentle and sweet voice)" + TARGET_TEXT,
        cfg_value=CFG_VALUE,
        inference_timesteps=INFERENCE_TIMESTEPS,
    )
    designed = np.asarray(designed, dtype=np.float32).squeeze()
    sf.write("output_voice_design.wav", designed, SAMPLE_RATE)
    print("✅ Saved output_voice_design.wav (designed voice, not your clone)")
    display(Audio(designed, rate=SAMPLE_RATE))
else:
    print("Extras skipped. Set RUN_EXTRAS = True to run style control / voice design.")

## 10. Troubleshooting notes

| Symptom | Fix |
|---|---|
| **No GPU detected** | Colab: *Runtime → Change runtime type → GPU*. CPU/MPS work but are slow. |
| **`ffmpeg` errors / can't read MP3** | Install ffmpeg (`!apt-get -y install ffmpeg` on Colab) or convert the sample to WAV first. |
| **Model download fails / gated** | `huggingface-cli login` (or set `HF_TOKEN`) and accept the model terms on its HF page. |
| **CUDA out of memory (loading)** | Set `LOAD_DENOISER=False`, restart runtime, or use a larger GPU (~8 GB needed). |
| **CUDA OOM (generation)** | Use a shorter reference clip, lower `INFERENCE_TIMESTEPS`, restart runtime. |
| **Clone doesn't resemble the sample** | Use a clean single-speaker clip; try `CLONE_MODE='ultimate'` with an accurate `PROMPT_TEXT`; raise `CFG_VALUE` (e.g. 2.5–3.0); try another reference. |
| **Noisy / breathy output** | Set `LOAD_DENOISER=True` **and** `DENOISE=True` to clean the reference. |
| **Numbers/abbreviations read wrong** | Set `NORMALIZE=True` (may require the text-normalization extra). |
| **Mispronounced / unstable runs** | Re-run (sampling varies); raise `INFERENCE_TIMESTEPS` a little; keep the reference clean. |

### Going further
- **Style control:** prefix the text with a `(natural-language style)` tag to
  steer emotion, pace, and expression while keeping the cloned timbre.
- **Voice design:** generate a brand-new voice from a description with no
  reference clip (see the extras cell).
- **Fine-tuning:** VoxCPM2 supports SFT and LoRA for production-grade,
  consistent cloning of one speaker. See the official
  [OpenBMB/VoxCPM](https://github.com/OpenBMB/VoxCPM) repository.